# 02 · KD con backbone descongelado

`ALFA` y `TEMPERATURA` no cambian respecto a `01`, para que la única variable entre ambos sea el descongelado.

In [ ]:
# En Kaggle clona el repo; en local no hace nada.
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/CCSandoval/ButterflyModeling.git"

if Path("/kaggle").exists():
    CLON = "/tmp/ButterflyModeling"
    if not Path(CLON).exists():                     # el repo es público: clonar no pide credenciales
        subprocess.run(["git", "clone", "--depth", "1", REPO, CLON], check=True)
    sys.path.insert(0, CLON)
    os.chdir(CLON)
    print("clon en", CLON, "·", subprocess.run(
        ["git", "log", "-1", "--format=%h %s"], cwd=CLON,
        capture_output=True, text=True).stdout.strip())

    try:
        from kaggle_secrets import UserSecretsClient

        os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        print("sin GITHUB_TOKEN: no se publicará")

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "corpus.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Raíz del proyecto:", ROOT)

In [ ]:
import platform
from datetime import datetime, timezone

import pandas as pd
import tensorflow as tf

from scripts import corpus, evalStats
from scripts import modelRegistry as registro
from scripts.architectures import IMG_SIZE, buildModel, descongelarCola, preprocessFn
from scripts.entrenamiento import callbacksEntrenamiento
from scripts.dataPipeline import buildAugmenter, loadSplit, preparar
from scripts.distillation import Destilador
from scripts.vizStyle import SERIE_1, SERIE_2, applyStyle, plotConfusionMatrix

estudianteInfo = registro.readJson(registro.OUTPUTS_DIR / "student.json")
if estudianteInfo is None:
    raise FileNotFoundError(
        "Falta outputs/student.json — corre antes 05_seleccion")

ARQUITECTURA = estudianteInfo["architecture"]
RUN_NAME = f"kd-respuesta-ft-{ARQUITECTURA.replace('_', '-')}"
NOTAS = "KD de respuesta con la cola del backbone del estudiante descongelada"
ROL = "destilado"

FRACCION_DESCONGELADA = 0.30
TEMPERATURA = 4.0
ALFA = 0.3

TAMANO = IMG_SIZE
LOTE = 32
EPOCAS = 100
EPOCAS_CALENTAMIENTO = 5
PACIENCIA = 8
TASA_APRENDIZAJE = 1e-4
SEMILLA = 42

tf.keras.utils.set_random_seed(SEMILLA)
corpus.materializar()

RUN_ID = registro.buildRunId(RUN_NAME)
RUN_DIR = registro.createRunDir(RUN_ID)
IMGS = RUN_DIR / "imgs"

print(f"run_id : {RUN_ID}")
print(f"TF     : {tf.__version__} · dispositivos: "
      f"{[d.device_type for d in tf.config.list_physical_devices()]}")

## 1 · Docente

Sale de `outputs/teacher.json`, escrito por `pausado/docentes/seleccion`.

In [ ]:
docenteInfo = registro.readJson(registro.OUTPUTS_DIR / "teacher.json")
if docenteInfo is None:
    raise FileNotFoundError("Falta outputs/teacher.json — corre antes pausado/docentes/seleccion")

ARQUITECTURA_DOCENTE = docenteInfo["architecture"]
docente = registro.modeloInferencia(docenteInfo["run_id"])
docente.trainable = False

pd.Series({
    "run_id": docenteInfo["run_id"],
    "arquitectura": ARQUITECTURA_DOCENTE,
    "test_accuracy": round(docenteInfo["test_accuracy"], 4),
    "test_macro_f1": round(docenteInfo["test_macro_f1"], 4),
    "parametros": docente.count_params(),
}).to_frame("docente")

## 2 · Datos

`train` sale crudo (0-255) y aumentado una sola vez; el `Destilador` preprocesa por dentro para cada modelo. `validate`/`test` van preprocesados para el estudiante, sin barajar.

In [ ]:
preprocessDocente = preprocessFn(ARQUITECTURA_DOCENTE)
preprocessEstudiante = preprocessFn(ARQUITECTURA)
augmentador = buildAugmenter()

dsTrainCrudo, clases = loadSplit("dataset/train", TAMANO, LOTE)
dsValCrudo, _ = loadSplit("dataset/validate", TAMANO, LOTE, shuffle=False)
dsTestCrudo, _ = loadSplit("dataset/test", TAMANO, LOTE, shuffle=False)

dsTrain = dsTrainCrudo.prefetch(tf.data.AUTOTUNE)
dsValDestilacion = dsValCrudo.prefetch(tf.data.AUTOTUNE)   # crudo: el Destilador preprocesa por dentro
dsVal = preparar(dsValCrudo, preprocessEstudiante)
dsTest = preparar(dsTestCrudo, preprocessEstudiante)

print(f"{len(clases)} clases")

## 3 · Estudiante

In [ ]:
estudiante = buildModel(ARQUITECTURA, len(clases), aumentar=False)

pd.Series({
    "estudiante": ARQUITECTURA,
    "parametros_estudiante": estudiante.count_params(),
    "parametros_docente": docente.count_params(),
    "razon_compresion": round(docente.count_params() / estudiante.count_params(), 1),
}).to_frame("valor")

## 4 · Pesos de clase

In [ ]:
pesosClase = corpus.classWeights()
pd.Series(
    {clases[i]: round(p, 3) for i, p in pesosClase.items()}
).sort_values(ascending=False).head(8).to_frame("peso")

## 5 · Entrenamiento con destilación

`val_loss` es la CE del estudiante, no la pérdida de destilación: así el EarlyStopping monitorea lo mismo que en los notebooks de candidatos. El docente corre en inferencia en cada paso — no se puede precomputar, la vista aumentada cambia cada época.

In [ ]:
from tensorflow.keras.optimizers import Adam

destilador = Destilador(
    docente, estudiante, preprocessDocente, preprocessEstudiante, TEMPERATURA, ALFA,
    aumentador=augmentador,
)
destilador.compile(optimizer=Adam(learning_rate=TASA_APRENDIZAJE))

inicio = datetime.now(timezone.utc)

calentamiento = {}
infoFt = None
if FRACCION_DESCONGELADA:
    calentamiento = destilador.fit(
        dsTrain,
        validation_data=dsValDestilacion,
        epochs=EPOCAS_CALENTAMIENTO,
        class_weight=pesosClase,
        verbose=2,
    ).history
    infoFt = descongelarCola(estudiante, FRACCION_DESCONGELADA)
    print(infoFt)
    destilador.compile(optimizer=Adam(learning_rate=TASA_APRENDIZAJE))

# reanudables: un corte cuesta una época, no el run entero
callbacks = callbacksEntrenamiento(RUN_NAME, PACIENCIA)

ajuste = destilador.fit(
    dsTrain,
    validation_data=dsValDestilacion,
    epochs=EPOCAS,
    callbacks=callbacks,
    class_weight=pesosClase,
    verbose=2,
)
fin = datetime.now(timezone.utc)

epocasCalentadas = len(calentamiento.get("loss", []))
historia = {}
for clave, valores in ajuste.history.items():
    previo = calentamiento.get(clave)
    historia[clave] = ([float(v) for v in previo] if previo else [None] * epocasCalentadas)
    historia[clave] += [float(v) for v in valores]
print(f"\n{len(historia['loss'])} épocas en {(fin - inicio).total_seconds() / 60:.1f} min")

## 6 · Curvas de entrenamiento

`loss` es la pérdida de destilación y `val_loss` la CE del estudiante, así que
no son directamente comparables entre sí (sí lo son época a época).

In [ ]:
applyStyle()
import matplotlib.pyplot as plt

epocas = range(1, len(historia["loss"]) + 1)
figura, ejes = plt.subplots(1, 2, figsize=(11, 4.2))

ejes[0].plot(epocas, historia["accuracy"], color=SERIE_1, label="Entrenamiento")
ejes[0].plot(epocas, historia["val_accuracy"], color=SERIE_2, label="Validación")
ejes[0].set(title=f"{RUN_NAME} · Exactitud", xlabel="Época", ylim=(0, 1))
ejes[0].legend(loc="lower right")

ejes[1].plot(epocas, historia["loss"], color=SERIE_1, label="Destilación (train)")
ejes[1].plot(epocas, historia["val_loss"], color=SERIE_2, label="CE (validación)")
ejes[1].set(title=f"{RUN_NAME} · Pérdida", xlabel="Época")
ejes[1].legend(loc="upper right")

figura.tight_layout()
figura.savefig(IMGS / "training_history.png", bbox_inches="tight")
plt.show()

## 7 · Evaluación

In [ ]:
metricas = {
    "test": evalStats.evaluar(estudiante, dsTest, clases),
    "validate": evalStats.evaluar(estudiante, dsVal, clases),
}

pd.DataFrame([
    {k: round(v, 4) if isinstance(v, float) else v
     for k, v in m.items()
     if k in ("num_samples", "accuracy", "top3_accuracy",
              "macro_f1", "weighted_f1", "macro_precision", "macro_recall")}
    for m in metricas.values()
], index=list(metricas.keys()))

## 8 · ¿Sirvió la destilación?

Contra el mismo estudiante sin docente (su run de entrenamiento) y contra el docente, que marca el techo.

In [ ]:
# el control debe compartir arquitectura Y régimen: si no, se compararía
# contra el estudiante congelado y el efecto del fine-tuning se atribuiría a la KD
controlRun = next(
    (r for r in registro.listRuns()
     if r["architecture"] == ARQUITECTURA
     and not r.get("teacher_run_id")
     and r.get("finetuning", 0.0) == FRACCION_DESCONGELADA),
    None,
)
if controlRun is None:
    raise ValueError(
        f"Falta el control: {ARQUITECTURA} sin destilar con "
        f"FRACCION_DESCONGELADA={FRACCION_DESCONGELADA}. Córrelo en 01_entrenamiento_mobilenetv3small.")

filas = [{
    "modelo": "docente",
    "arquitectura": ARQUITECTURA_DOCENTE,
    "test_accuracy": docenteInfo["test_accuracy"],
    "test_macro_f1": docenteInfo["test_macro_f1"],
    "parametros": docente.count_params(),
}]
if controlRun:
    filas.append({
        "modelo": "control (sin KD)",
        "arquitectura": ARQUITECTURA,
        "test_accuracy": controlRun["test_accuracy"],
        "test_macro_f1": controlRun["test_macro_f1"],
        "parametros": estudiante.count_params(),
    })
else:
    print("Sin run de control registrado — corre antes los notebooks de entrenamiento")
filas.append({
    "modelo": "estudiante (KD)",
    "arquitectura": ARQUITECTURA,
    "test_accuracy": metricas["test"]["accuracy"],
    "test_macro_f1": metricas["test"]["macro_f1"],
    "parametros": estudiante.count_params(),
})

comparacion = pd.DataFrame(filas)
if controlRun:
    delta = metricas["test"]["macro_f1"] - controlRun["test_macro_f1"]
    brecha = docenteInfo["test_macro_f1"] - controlRun["test_macro_f1"]
    print(f"Delta macro-F1 por destilar: {delta:+.4f}")
    if brecha > 0:
        print(f"Recupera el {delta / brecha:.1%} de la brecha estudiante-docente")
comparacion

## 9 · Matriz de confusión

In [ ]:
_ = plotConfusionMatrix(
    metricas["test"],
    f"Matriz de confusión · prueba · {RUN_NAME}",
)
plt.savefig(IMGS / "confusion_test.png", bbox_inches="tight")
plt.show()

## 10 · Confusiones principales

In [ ]:
confusiones = evalStats.confusionesPrincipales(metricas["test"])
pd.DataFrame(confusiones).head(10)

## 11 · Versionado

In [ ]:
estudiante.save(RUN_DIR / "model.keras")
registro.writeJson(RUN_DIR / "history.json", historia)
registro.writeJson(RUN_DIR / "metrics.json", {
    "run_id": RUN_ID,
    "class_names": clases,
    "splits": metricas,
    "top_confusions": {
        s: evalStats.confusionesPrincipales(m) for s, m in metricas.items()
    },
})

configuracion = {
    "run_id": RUN_ID,
    "name": RUN_NAME,
    "dataset": corpus.manifiesto()["creado"],
    "architecture": ARQUITECTURA,
    "role": ROL,
    "notes": NOTAS,
    "num_classes": len(clases),
    "distillation": {
        "tipo": "respuesta",
        "teacher_run_id": docenteInfo["run_id"],
        "teacher_architecture": ARQUITECTURA_DOCENTE,
        "temperatura": TEMPERATURA,
        "finetuning": FRACCION_DESCONGELADA,
        "alfa": ALFA,
    },
    "hyperparameters": {
        "img_size": list(TAMANO),
        "batch_size": LOTE,
        "epochs_budget": EPOCAS,
        "epochs_warmup": EPOCAS_CALENTAMIENTO if FRACCION_DESCONGELADA else 0,
        "epochs_ran": len(historia["loss"]),
        "learning_rate": TASA_APRENDIZAJE,
        "early_stopping_patience": PACIENCIA,
        "class_weights": "balanced",
        "seed": SEMILLA,
        "backbone": f"{ARQUITECTURA}/imagenet (congelado)",
    },
    "environment": {
        "commit": registro.commitActual(),
        "python": platform.python_version(),
        "tensorflow": tf.__version__,
        "platform": platform.platform(),
        "acelerador": next((d.device_type for d in tf.config.list_physical_devices()
                            if d.device_type != "CPU"), "cpu"),
    },
    "started_at": inicio.isoformat(),
    "finished_at": fin.isoformat(),
    "training_minutes": round((fin - inicio).total_seconds() / 60, 1),
}
registro.writeJson(RUN_DIR / "run.json", configuracion)

registro.registrar(RUN_ID)

print(f"Run guardado en {RUN_DIR}")

if Path("/kaggle").exists() and os.environ.get("GITHUB_TOKEN"):
    from scripts.kaggle import publicarRun

    publicarRun(RUN_ID)


## Resumen

In [ ]:
pd.Series({
    "run_id": RUN_ID,
    "estudiante": ARQUITECTURA,
    "docente": ARQUITECTURA_DOCENTE,
    "temperatura": TEMPERATURA,
    "alfa": ALFA,
    "epocas": len(historia["loss"]),
    "minutos": configuracion["training_minutes"],
    "test_accuracy": round(metricas["test"]["accuracy"], 4),
    "test_top3": round(metricas["test"]["top3_accuracy"], 4),
    "test_macro_f1": round(metricas["test"]["macro_f1"], 4),
    "validate_accuracy": round(metricas["validate"]["accuracy"], 4),
    "validate_macro_f1": round(metricas["validate"]["macro_f1"], 4),
}).to_frame("valor")